# 04 · The Feedback Loop (Novelty, part 2) — The Engagement Trap

Real recommenders are retrained on clicks they caused. This notebook simulates that loop and asks two questions:
1. **How much of the catalogue does a click-trained recommender actually show people over time?**
2. **Can the click metric (CTR, the *proxy*) tell a narrow system apart from a broader one?**

This is the Goodhart angle of the project: if the number you optimise can't see a side effect, optimising it gives you no warning about that side effect. We compare plain SVD against the diversity-aware policies from notebook 03.

### Simulation design
1. **Hidden "true user" (oracle) — Exp 2, Logistic Regression.** Predicts P(user likes movie) from
   - user-genre affinity × movie genres (one feature per genre),
   - user mean rating, movie mean rating (damped), log movie popularity.
   It is fitted on the training split and its **AUC is checked on the held-out test split**. The recommender never sees it.
2. **Start:** every user has 10 random movies from their real training history.
3. **Each round:**
   - the recommender (SVD on the 0/1 click matrix) is retrained on everything clicked so far,
   - each user is shown 10 movies they have not clicked yet, chosen by one of 4 **policies**,
   - the user clicks the movie at position *p* with probability `P_true × 1/log2(p + 2)` (people click top slots more),
   - clicks are added to the training log.
4. **Policies:** plain SVD, MMR, K-Means cluster cap, DBSCAN cluster cap (λ and caps from notebook 03 if its output is attached).

### Metrics tracked every round
| Metric | Meaning |
|---|---|
| **CTR** | clicks / impressions — the **proxy** the system can see |
| **TrueUtility** | mean hidden P(like) of the shown movies |
| **Discovery** | share of each user's 50 best-matching movies (by the oracle) that have *ever* been shown — the long-run goal |
| **CumCoverage** | share of the catalogue shown at least once so far |
| **ExposureGini** | inequality of cumulative exposure across movies |
| **ILD** | genre diversity inside each list |
| **AvgPop** | mean popularity of shown movies |

### Why the oracle does not use SVD
An earlier draft built the oracle from the same SVD factors as the SVD recommender. That makes the experiment circular: the recommender would share the "true" model's structure, and an in-sample AUC would overstate how realistic the oracle is. Here the oracle uses **no latent factors at all**, and we report held-out AUC. Note this oracle is built on genre affinity, so it tends to reward *staying* in a user's favourite genres — that works **against** diversity policies, which makes any benefit we find for them conservative.

In [ ]:
import os, sys, glob, json, time
# On Kaggle, etlib.py comes from an attached dataset; locally it sits next to this notebook.
for hit in glob.glob("/kaggle/input/**/etlib.py", recursive=True):
    sys.path.insert(0, os.path.dirname(hit))
    break
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import etlib as et

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})
OUT = et.output_dir()
DATA_DIR = et.find_data_dir()
ratings, items, GENRES = et.load_movielens(DATA_DIR)  # movies with >= 5 ratings
G = et.genre_matrix(items, GENRES)
print("data:", DATA_DIR, "| outputs:", OUT)
print(len(ratings), "ratings |", ratings.user.nunique(), "users |", ratings.item.nunique(), "movies")


def save_fig(fig, name):
    fig.savefig(f"{OUT}/figures/{name}.png", dpi=150, bbox_inches="tight")

## 1. Configuration

In [ ]:
SEEDS = [0, 1, 2]
N_ROUNDS = 30
K = 10
SVD_K = 20
POOL = 50
START_PER_USER = 10
LAM = 0.7
CAP = {"kmeans_cap": 2, "dbscan_cap": 2}

# use the lambda / caps chosen on validation in notebook 03 if its output is available
hits = glob.glob("/kaggle/input/**/rerank.json", recursive=True) + glob.glob(f"{OUT}/demo_data/rerank.json")
if hits:
    cfg = json.load(open(hits[0]))
    LAM = cfg["lambda"]
    CAP = {"kmeans_cap": cfg["cap"]["kmeans"], "dbscan_cap": cfg["cap"]["dbscan"]}
    print("using settings from", hits[0])
else:
    print("notebook 03 output not found - using defaults")
print("lambda =", LAM, "| caps =", CAP)

## 2. The hidden oracle (Logistic Regression)

In [ ]:
oracles, oracle_rows = {}, []
for seed in SEEDS:
    train, test = et.split_per_user(ratings, 0.2, seed)
    oracle = et.LogisticOracle().fit(train, G)
    oracles[seed] = (train, test, oracle)
    oracle_rows.append({"seed": seed, "AUC_train": oracle.auc(train), "AUC_heldout_test": oracle.auc(test)})
oracle_auc = pd.DataFrame(oracle_rows)
oracle_auc.to_csv(f"{OUT}/results/04_oracle_auc.csv", index=False)
oracle_auc.round(4)

In [ ]:
# sanity check: the fast full-matrix formula equals sklearn's predict_proba
train0, test0, oracle0 = oracles[0]
P0 = oracle0.prob_matrix()
sample = train0.sample(2000, random_state=0)
u, i = sample.user.to_numpy(), sample.item.to_numpy()
assert np.allclose(P0[u, i], oracle0.lr.predict_proba(oracle0._features(u, i))[:, 1])
print("prob_matrix matches predict_proba. P_true shape:", P0.shape)

In [ ]:
from sklearn.calibration import calibration_curve

fig, axes = plt.subplots(1, 3, figsize=(17, 4.3))
p_test = P0[test0.user.to_numpy(), test0.item.to_numpy()]
frac, mean_pred = calibration_curve(test0.rating >= 4, p_test, n_bins=10)
axes[0].plot(mean_pred, frac, "o-", label="oracle")
axes[0].plot([0, 1], [0, 1], "k--", label="perfect")
axes[0].set(xlabel="predicted P(like)", ylabel="observed like rate", title="Calibration on held-out test")
axes[0].legend()

coef = pd.Series(oracle0.lr.coef_.ravel(), index=[f"aff×{g}" for g in GENRES] + ["user_mean", "item_mean", "log_pop"])
coef.sort_values().plot.barh(ax=axes[1], color="slateblue", title="Oracle coefficients (seed 0)")

pop_tr0 = et.item_counts(train0)
axes[2].scatter(np.log1p(pop_tr0), P0.mean(axis=0), s=5, alpha=0.4)
axes[2].set(xlabel="log(1 + popularity)", ylabel="mean P(like) over users",
            title="Does the oracle itself favour popular movies?")
fig.tight_layout()
save_fig(fig, "04_oracle")
plt.show()
print("corr(log popularity, mean P_true):", round(np.corrcoef(np.log1p(pop_tr0), P0.mean(axis=0))[0, 1], 3))

If the oracle strongly prefers popular movies, then some popularity concentration is *justified* by user taste — the interesting question is whether the loop concentrates exposure **more** than taste alone explains, and whether users miss movies they would have liked (Discovery).

## 3. Run the feedback loop

In [ ]:
histories = []
start = time.time()
for seed in SEEDS:
    train, test, oracle = oracles[seed]
    P_true = oracle.prob_matrix()
    start_log = et.seed_log(train, START_PER_USER, seed)
    popularity = et.item_counts(train)

    # clusters come from the round-0 recommender's movie embeddings and stay fixed
    _, vecs = et.implicit_svd(start_log, SVD_K, seed)
    # (movies nobody has clicked yet have a zero embedding, so they are left unclustered)
    active = start_log.sum(axis=0) > 0
    labels = {"kmeans_cap": et.choose_kmeans(vecs, seed=seed, active=active)[0],
              "dbscan_cap": et.choose_dbscan(vecs, active=active)[0]}

    for policy in et.POLICIES:
        h = et.simulate(policy, P_true, start_log, G, popularity, n_rounds=N_ROUNDS, k=K,
                        svd_k=SVD_K, pool_size=POOL, lam=LAM, cap=CAP.get(policy, 2),
                        labels=labels.get(policy), seed=seed)
        h["policy"], h["seed"] = policy, seed
        histories.append(h)
        print(f"seed {seed} | {policy:10s} | {time.time() - start:5.0f}s")

sim = pd.concat(histories, ignore_index=True)
sim.to_csv(f"{OUT}/results/04_simulation_rounds.csv", index=False)

## 4. How the metrics evolve (mean ± std over seeds)

In [ ]:
METRICS = ["CTR", "TrueUtility", "Discovery", "CumCoverage", "ExposureGini", "ILD", "AvgPop"]
mean = sim.groupby(["policy", "round"])[METRICS].mean()
std = sim.groupby(["policy", "round"])[METRICS].std()

fig, axes = plt.subplots(2, 4, figsize=(20, 8.5))
for ax, m in zip(axes.ravel(), METRICS):
    for policy in et.POLICIES:
        mu, sd = mean.loc[policy][m], std.loc[policy][m]
        ax.plot(mu.index, mu, label=policy)
        ax.fill_between(mu.index, mu - sd, mu + sd, alpha=0.2)
    ax.set(title=m, xlabel="round")
axes[0, 0].legend()

final_round = sim[sim["round"] == N_ROUNDS]
ax = axes[1, 3]
for policy in et.POLICIES:
    f = final_round[final_round.policy == policy]
    ax.scatter(f["CTR"], f["Discovery"], label=policy)
ax.set(xlabel="CTR in final round (proxy)", ylabel="Discovery after all rounds",
       title="Proxy vs long-run goal (one dot per seed)")
fig.tight_layout()
save_fig(fig, "04_feedback_loop")
plt.show()

## 5. Final-round comparison and change over time

In [ ]:
table, agg = et.mean_std_table(final_round.drop(columns="round").to_dict("records"), group="policy")
table.to_csv(f"{OUT}/results/04_final_round_mean_std.csv")
display(table)

first = sim[sim["round"] == 1].groupby("policy")[METRICS].mean()
last = final_round.groupby("policy")[METRICS].mean()
change = (last - first).loc[et.POLICIES]
change.to_csv(f"{OUT}/results/04_change_round1_to_final.csv")
print("Change from round 1 to round", N_ROUNDS, "(mean over seeds):")
change.round(4)

**How to read the results**
- **Reach:** how much of the catalogue has plain SVD shown after all rounds (CumCoverage), and how much does it grow after round 1? A flat curve means exposure is *stuck*, even if it is not shrinking.
- **Is the proxy blind?** Compare CTR and CumCoverage across policies. If two policies have about the same CTR but very different coverage, the click metric gives no signal about how narrow the system is.
- **Cost of diversity:** do the diversity policies lose CTR or TrueUtility (beyond the seed-to-seed std)?
- **Discovery:** does showing more of the catalogue help users reach more of their best-matching movies?
- **K-Means vs DBSCAN cap:** which clustering makes the cap useful?
- CTR and TrueUtility move together *by construction* (clicks are drawn from P_true), and clicked movies are never shown again. So both fall over the rounds for every policy as users use up their best matches, and AvgPop falls as popular movies are clicked early. These trends are properties of the simulator, not evidence for or against the loop narrowing exposure.

**Simulation assumptions to state as limitations:** the oracle is a model of taste, not real users; P(like) is estimated only from movies people chose to rate (selection bias); tastes do not change over time; users always return and always see 10 items; the position-bias curve is assumed, not measured; clusters are computed once at round 0.

## 6. Export for the demo site

In [ ]:
export = {"rounds": list(range(1, N_ROUNDS + 1)), "metrics": METRICS,
          "config": {"seeds": SEEDS, "k": K, "svd_k": SVD_K, "pool": POOL, "lambda": LAM, "caps": CAP,
                     "start_per_user": START_PER_USER},
          "oracle_auc": [{k: et.json_num(v) for k, v in r.items()} for r in oracle_rows],
          "policies": {p: {m: {"mean": [et.json_num(v) for v in mean.loc[p][m]],
                               "std": [et.json_num(v) for v in std.loc[p][m]]} for m in METRICS}
                       for p in et.POLICIES}}
et.save_json(export, f"{OUT}/demo_data/simulation.json")
print("saved", f"{OUT}/demo_data/simulation.json", "| total time", round(time.time() - start), "s")